# Explicação de `pacote.py`: espectro do fóton de saída de um sítio de Brod-Combes com banho em estado de Fock

Este notebook percorre `pacote.py` linha por linha. Para cada variável, expressão e gráfico, ele diz o que aquilo
representa e escreve a fórmula em LaTeX. As células de código reproduzem o cálculo do script. A única diferença é que
**não salvam** a figura, para não sobrescrever `figura4_espectro_*.svg`.

**O sistema.** É o mesmo de `main.py` (ver `main_explicado.ipynb`). Um átomo de dois níveis está acoplado a um guia de onda
com taxa $\gamma$ e a **um único** modo harmônico (a "mola", ou banho) de frequência $\omega$, que começa no estado de
Fock $|n\rangle$. Um fóton incide sobre o átomo.

**O que muda em relação a `main.py`.** `main.py` olha só o canal *limpo*, $|t_{nn}|^2$. Já `pacote.py` olha **todos os
canais**: a mola pode terminar em qualquer $|m\rangle$, e o fóton sai com a frequência deslocada pela energia que deixou na
mola ou retirou dela. O resultado é o **espectro do fóton de saída**:

$$
S_{\text{out}}(\omega_a) = \sum_m \big|\xi(\nu_m)\big|^2\,\big|t_{mn}(\nu_m)\big|^2,
\qquad \nu_m = \omega_a + (m-n)\,\omega .
$$

**A figura** tem 2 colunas e 2 linhas:

| | Mola rápida (esquerda) | Mola lenta (direita) |
|---|---|---|
| **Em cima** | fóton monocromático: linhas discretas | fóton monocromático: linhas discretas |
| **Embaixo** | fóton com largura $\sigma$: picos alargados | fóton com largura $\sigma$: picos alargados |

Os painéis de cima e de baixo da mesma coluna têm o **mesmo eixo $x$** (`sharex='col'`), a frequência de saída $\omega_a/\gamma$.

## 1. Importações

| Nome | Origem | Uso |
|---|---|---|
| `np` | `numpy` | arrays, `sqrt`, `exp`, `linspace`, `zeros_like` |
| `eval_genlaguerre` | `scipy.special` | polinômio de Laguerre generalizado $L_n^{(\alpha)}(x)$ |
| `factorial` | `scipy.special` | fatorial $n!$ (em ponto flutuante) |
| `plt` | `matplotlib.pyplot` | figuras |

In [ ]:
import numpy as np
from scipy.special import eval_genlaguerre, factorial
import matplotlib.pyplot as plt

## 2. Parâmetros do átomo

Todas as frequências são medidas em unidades de $\gamma$. Por isso `gam = 1`.

| Variável | Símbolo | Significado |
|---|---|---|
| `gam` | $\gamma$ | taxa de emissão do átomo no guia. Fixa a unidade, e a meia-largura da linha é $\gamma/2$ |
| `Delta` | $\Delta$ | dessintonia do átomo, isto é, onde a linha estaria sem o banho |

In [ ]:
gam   = 1.0      # taxa de emissao do atomo no guia
Delta = 0.0      # dessintonia do atomo

## 3. Parâmetros do banho e do fóton

| Variável | Símbolo | Valor | Significado |
|---|---|---|---|
| `n` | $n$ | $1$ | quanta iniciais na mola, $|n\rangle$. Com $n \ge 1$ a mola tem um quantum para **dar** ao fóton, o que abre o canal $m = n-1$ (linha "azul", de maior energia) |
| `S` | $S$ | $0.25$ | fator de Huang-Rhys, $S = \beta^2 = (2g/\omega)^2$. É o mesmo nos dois casos |
| `sig` | $\sigma$ | $0.3\gamma$ | largura espectral do pacote do fóton, usada só nos gráficos de baixo |
| `K` | $K$ | $60$ | corte da soma sobre os níveis $k$ do oscilador deslocado: $k = 0, \dots, K-1$ |
| `dms` | $m-n$ | $-1, 0, \dots, 4$ | canais mostrados, isto é, $m = n-1, \dots, n+4$ |
| `casos` | | | lista de tuplas `(titulo, omega, xlim)` |

Como em `main.py`, fixamos $S$ e mudamos só $\omega$. O acoplamento sai de $S$ (seção 6):

$$
\beta = \frac{2g}{\omega}, \qquad S = \beta^2 \quad\Longrightarrow\quad g = \frac{\sqrt{S}\,\omega}{2}.
$$

### Lista `casos`

| Caso | $\omega/\gamma$ | `xlim` (eixo $\omega_a/\gamma$) | Regime |
|---|---|---|---|
| Mola rápida | $3.0$ | $(-15.5,\ 3.5)$ | $\omega \gg \gamma, \sigma$: cada canal $m$ sai numa cor bem separada |
| Mola lenta | $0.1$ | $(-1.6,\ 1.6)$ | $\omega \ll \gamma, \sigma$: os canais ficam a $0.1\gamma$ um do outro e se sobrepõem |

Os limites de `xlim` foram escolhidos para mostrar todas as linhas $\omega_a = \nu_0 - (m-n)\,\omega$ com $m-n \in$ `dms`.
Na mola rápida, elas vão de $+3$ ($m-n=-1$) a $-12$ ($m-n=+4$).

> **Cuidado ao mudar `n` ou `dms`.** É preciso ter $m = n + (m-n) \ge 0$ para todo valor de `dms`. Com `n = 0`, por exemplo,
> o primeiro valor de `dms` tem de ser `0`, porque não existe $|m=-1\rangle$.

### Dicionários de estilo

| Variável | Significado |
|---|---|
| `cores` | cor de cada canal $m-n$: azul ($-1$), preto ($0$), vermelho ($+1$), laranja ($+2$), roxo ($+3$), verde ($+4$) |
| `nomes` | rótulo de cada canal na legenda |

Nomes dos canais, do ponto de vista do **fóton**:

- $m = n-1$: a mola perde um quantum e o fóton **ganha** energia $\omega$ (linha anti-Stokes, deslocada para o azul).
- $m = n$: espalhamento elástico, o fóton sai **"limpo"** com a mesma cor.
- $m = n+1, n+2, \dots$: a mola ganha quanta e o fóton **perde** energia (linhas Stokes, deslocadas para o vermelho).

In [ ]:
n     = 1        # quanta iniciais na mola (n>=1 permite linha "azul")
S     = 0.25     # fator de Huang-Rhys S=(2g/omega)^2, igual nos dois casos
sig   = 0.3     # largura espectral do pacote do foton (graficos de baixo)
K     = 60       # corte da soma sobre os niveis k do oscilador deslocado
dms   = range(-1, 5)   # valores de m-n mostrados (m = n-1, ..., n+3)
casos = [("Mola rapida", 3.0, (-15.5, 3.5)),  # (titulo, omega, limites do eixo x)
         ("Mola lenta",  0.1, (-1.6, 1.6))]

cores = {-1: 'tab:blue', 0: 'k', 1: 'tab:red', 2: 'tab:orange', 3: 'tab:purple', 4: 'tab:green'}
nomes = {-1: 'm=n-1 (ganha energia)', 0: 'm=n (limpo)', 1: 'm=n+1 (perde energia)',
         2: 'm=n+2', 3: 'm=n+3', 4: 'm=n+4'}

## 4. Fator de Franck-Condon — `FC(k, n, beta)`

É o elemento de matriz do **operador deslocamento** $D(\beta) = e^{\beta(a^\dagger - a)}$ entre estados de Fock:

$$
F_{kn} \equiv \langle k | D(\beta) | n \rangle .
$$

Ele dá a amplitude de a mola passar de $|n\rangle$ (poço do átomo no estado fundamental) para o nível $|k\rangle$ do poço
deslocado (átomo excitado).

**Caso $k \ge n$** ($\beta$ real):

$$
F_{kn} = \sqrt{\frac{n!}{k!}}\;\beta^{\,k-n}\;e^{-\beta^2/2}\;L_n^{(k-n)}\!\left(\beta^2\right),
$$

em que $L_n^{(\alpha)}$ é o polinômio de Laguerre generalizado (`eval_genlaguerre(n, k-n, beta**2)`).

**Caso $k < n$**, pela simetria $D(\beta)^\dagger = D(-\beta)$:

$$
F_{kn} = (-1)^{\,n-k}\,F_{nk}.
$$

**Regra de soma:** $\displaystyle\sum_{k=0}^{\infty} F_{kn}^2 = 1$, porque $D$ é unitário. Com $\beta = 0.5$ e $n = 1$, os pesos
$F_{k1}^2$ valem $\approx 0.19,\ 0.44,\ 0.30,\ 0.06,\ 0.007$ para $k = 0, \dots, 4$, de modo que $K = 60$ sobra com folga.

In [ ]:
def FC(k, n, beta):
    """Fator de Franck-Condon <k|D(beta)|n>, beta real."""
    if k >= n:
        return (np.sqrt(factorial(n) / factorial(k)) * beta**(k - n)
                * np.exp(-beta**2 / 2) * eval_genlaguerre(n, k - n, beta**2))
    return (-1)**(n - k) * FC(n, k, beta)

## 5. Amplitude de espalhamento — `t_mn(m, n, nu, w, g)`

A amplitude para um fóton de frequência $\nu$ (= `nu`) passar pelo átomo levando a mola de $|n\rangle$ para $|m\rangle$ é

$$
t_{mn}(\nu) = \delta_{mn} \;-\; \gamma \sum_{k=0}^{K-1}
\frac{F_{km}\,F_{kn}}
{\dfrac{\gamma}{2} + i\left(\Delta - \dfrac{4g^2}{\omega} + (k-n)\,\omega - \nu\right)} .
$$

| Termo | Variável no código | Significado |
|---|---|---|
| $\delta_{mn}$ | `1.0 if m == n else 0.0` | parte do fóton que passa sem interagir. Só existe no canal elástico |
| $k$ | `k` | nível intermediário do oscilador **deslocado**, com o átomo excitado |
| $F_{kn}$ | `FC(k, n, beta)` | absorção: $\lvert g,n\rangle \to \lvert e,\tilde k\rangle$ |
| $F_{km}$ | `FC(k, m, beta)` | emissão: $\lvert e,\tilde k\rangle \to \lvert g,m\rangle$ |
| $\gamma/2$ | `gam / 2` | meia-largura da ressonância |
| $4g^2/\omega = S\omega$ | `4 * g**2 / w` | deslocamento polarônico |
| $(k-n)\,\omega$ | `(k - n) * w` | energia de vibração trocada na absorção |
| denominador | `den` | `gam / 2 + 1j * (Delta - 4*g**2/w + (k-n)*w - nu)` |

Diferença em relação a `main.py`: aqui `gam`, `Delta` e `K` não são argumentos. A função os lê das variáveis globais
definidas nas seções 2 e 3.

**Unitariedade.** Para cada $\nu$, a probabilidade se divide entre os canais sem perdas:

$$
\sum_{m=0}^{\infty} \big|t_{mn}(\nu)\big|^2 = 1 .
$$

In [ ]:
def t_mn(m, n, nu, w, g):
    """Amplitude de o foton entrar com cor nu e a mola ir de |n> para |m>."""
    beta = 2 * g / w
    s = 0.0
    for k in range(K):
        den = gam / 2 + 1j * (Delta - 4 * g**2 / w + (k - n) * w - nu)
        s += FC(k, m, beta) * FC(k, n, beta) / den
    return (1.0 if m == n else 0.0) - gam * s

## 6. Conservação de energia e o espectro de saída

### Cor de saída

Se o fóton entra com frequência $\nu_a$ e a mola vai de $|n\rangle$ para $|m\rangle$, a energia $(m-n)\,\omega$ fica na mola.
O fóton sai então com

$$
\omega_a = \nu_a - (m-n)\,\omega
\qquad\Longleftrightarrow\qquad
\nu_a = \omega_a + (m-n)\,\omega .
$$

### Pacote de entrada

O fóton incidente tem amplitude espectral $\xi(\nu)$, centrada em $\nu_0$ e com largura $\sigma$. O script usa a densidade
espectral gaussiana normalizada (`xi2`):

$$
|\xi(\nu)|^2 = \frac{1}{\sqrt{2\pi\sigma^2}}\,\exp\!\left[-\frac{(\nu-\nu_0)^2}{2\sigma^2}\right],
\qquad \int |\xi(\nu)|^2\,d\nu = 1 .
$$

Aqui $\sigma$ é o desvio padrão da **intensidade** $|\xi|^2$, e o pico vale $1/\sqrt{2\pi\sigma^2} \approx 1.33$ para $\sigma = 0.3$.

### Espectro de saída

Cada componente $\nu$ do pacote vira

$$
\xi(\nu)\,|\nu\rangle|n\rangle \;\longrightarrow\; \sum_m \xi(\nu)\,t_{mn}(\nu)\,\big|\nu-(m-n)\omega\big\rangle\,|m\rangle .
$$

Os estados $|m\rangle$ da mola são ortogonais, então os canais **não interferem** ao traçar a mola. Isso quer dizer que se
somam probabilidades, e não amplitudes. Trocando a variável para a cor de saída $\omega_a$ (o jacobiano vale 1):

$$
S_{\text{out}}(\omega_a) = \sum_m S_m(\omega_a),
\qquad
S_m(\omega_a) = \big|\xi(\omega_a + (m-n)\omega)\big|^2\,\big|t_{mn}(\omega_a + (m-n)\omega)\big|^2 .
$$

Pela unitariedade, a área total se conserva:

$$
\int S_{\text{out}}(\omega_a)\,d\omega_a = \int |\xi(\nu)|^2 \sum_m |t_{mn}(\nu)|^2\,d\nu = 1 .
$$

### Fóton monocromático (limite $\sigma \to 0$)

Com $|\xi(\nu)|^2 \to \delta(\nu-\nu_0)$, o espectro vira um conjunto de linhas discretas:

$$
S_{\text{out}}(\omega_a) = \sum_m P_m\,\delta\big(\omega_a - [\nu_0 - (m-n)\omega]\big),
\qquad P_m = \big|t_{mn}(\nu_0)\big|^2,
\qquad \sum_m P_m = 1 .
$$

São essas as linhas dos gráficos de cima.

## 7. Laço principal: as variáveis de cada painel

Para cada caso `(titulo, w, xlim)`, com índice de coluna `j`:

| Variável | Expressão | Significado |
|---|---|---|
| `g` | $g = \sqrt{S}\,\omega/2$ | acoplamento que realiza o $S$ escolhido |
| `nu0` | $\nu_0 = \Delta$ | centro do fóton incidente |
| `xi2` | $\lvert\xi(\nu)\rvert^2$ | função (`lambda`) com a densidade gaussiana da seção 6 |

**Sobre `nu0`.** A linha `nu0 = Delta #- 4 * g**2 / w` tem o deslocamento polarônico **comentado**. Por isso o fóton fica
centrado na linha **nua** do átomo ($\nu_0 = \Delta = 0$), e não no vale de zero fônon $\Delta - S\omega$. Na mola rápida isso
põe o fóton $S\omega = 0.75\gamma$ fora da ZPL. Na mola lenta, a diferença é só $S\omega = 0.025\gamma$. Para centrar o
fóton na ZPL, basta tirar o `#`.

Valores numéricos:

| Caso | $\omega$ | $g$ | $\beta$ | $S\omega$ (ZPL em $\Delta - S\omega$) |
|---|---|---|---|---|
| Mola rápida | $3.0$ | $0.75$ | $0.5$ | $0.75$ |
| Mola lenta | $0.1$ | $0.025$ | $0.5$ | $0.025$ |

### Painéis de cima — fóton com cor exata (`ax[0, j]`)

Para cada `d` em `dms`:

| Variável | Expressão | Significado |
|---|---|---|
| `d` | $m - n$ | canal |
| `m` | $m = n + d$ | estado final da mola |
| `p` | $P_m = \lvert t_{mn}(\nu_0)\rvert^2$ | probabilidade (altura) da linha |
| `wa` | $\omega_a = \nu_0 - d\,\omega$ | cor de saída (posição) da linha |
| `soma` | $\sum_{d \in \texttt{dms}} P_{n+d}$ | soma das probabilidades. Deve dar $\approx 1$ |

Cada linha é desenhada com `vlines` de $0$ até $P_m$, na cor `cores[d]`. O eixo $y$ é a **probabilidade**, de $0$ a $1.05$.

### Painéis de baixo — fóton com largura $\sigma$ (`ax[1, j]`)

| Variável | Expressão | Significado |
|---|---|---|
| `wa` | $\omega_a \in$ `xlim` | eixo $x$ com 2001 pontos (agora é um array, e não mais a posição de uma linha) |
| `nus` | $\nu = \omega_a + d\,\omega$ | cor de entrada que leva à saída $\omega_a$ no canal $d$ |
| `Sm` | $S_m(\omega_a) = \lvert\xi(\nu)\rvert^2\,\lvert t_{mn}(\nu)\rvert^2$ | contribuição do canal $m$ ao espectro de saída |
| `total` | $S_{\text{out}}(\omega_a) = \sum_d S_{n+d}(\omega_a)$ | espectro de saída total |

Curvas desenhadas:

| Curva | Estilo | Expressão |
|---|---|---|
| canal $m$ | linha cheia, cor `cores[d]` | $S_m(\omega_a)$ |
| espectro de entrada | cinza pontilhado | $\lvert\xi(\omega_a)\rvert^2$ (o que sairia se o átomo não existisse) |
| espectro de saída total | verde tracejado | $S_{\text{out}}(\omega_a)$ |

O eixo $y$ é a **densidade espectral** (unidades de $1/\gamma$). Note que o canal $m = n+4$ e a curva total usam a mesma
cor, `tab:green`.

## 8. Verificações numéricas (não estão em `pacote.py`)

1. **Regra de soma de Franck-Condon:** $\sum_{k<K} F_{kn}^2 \approx 1$. Mostra que o corte `K` basta.
2. **Linhas de cima:** $\sum_{m-n \in \texttt{dms}} P_m \approx 1$. O que falta para 1 é o peso dos canais fora de `dms`.
3. **Área de baixo:** $\int |\xi|^2\,d\omega_a$ e $\int S_{\text{out}}\,d\omega_a$ dentro da janela `xlim`. Os dois devem ser
   $\approx 1$, salvo a parte da gaussiana que cai fora da janela e o peso dos canais fora de `dms`.

In [ ]:
for titulo, w, xlim in casos:
    g = np.sqrt(S) * w / 2
    nu0 = Delta
    xi2 = lambda v: np.exp(-(v - nu0)**2 / (2 * sig**2)) / np.sqrt(2 * np.pi * sig**2)
    soma_FC = sum(FC(k, n, 2 * g / w)**2 for k in range(K))
    soma_P  = sum(abs(t_mn(n + d, n, nu0, w, g))**2 for d in dms)
    wa = np.linspace(xlim[0], xlim[1], 2001)
    total = sum(xi2(wa + d * w) * np.array([abs(t_mn(n + d, n, v, w, g))**2 for v in wa + d * w])
                for d in dms)
    print(f'{titulo:12s}: sum_k F_kn^2 = {soma_FC:.6f}   sum_m P_m = {soma_P:.5f}   '
          f'int xi^2 = {np.trapezoid(xi2(wa), wa):.5f}   int S_out = {np.trapezoid(total, wa):.5f}')

## 9. A figura

O código abaixo é o laço de `pacote.py`. Ele imprime, para cada caso, a posição $\omega_a$ e a probabilidade $P_m$ de cada
linha, e no fim a soma.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 6.6), sharex='col')

for j, (titulo, w, xlim) in enumerate(casos):
    g = np.sqrt(S) * w / 2                  # acoplamento que da o S escolhido
    nu0 = Delta #- 4 * g**2 / w              # foton centrado no vale do meio
    xi2 = lambda v: np.exp(-(v - nu0)**2 / (2 * sig**2)) / np.sqrt(2 * np.pi * sig**2)

    # ---- em cima: foton monocromatico em nu0 -> linhas ----
    print(titulo, ' nu_a =', round(nu0, 3))
    soma = 0.0
    for d in dms:
        m = n + d
        p = abs(t_mn(m, n, nu0, w, g))**2   # probabilidade da linha
        wa = nu0 - d * w                     # cor de saida dessa linha
        soma += p
        ax[0, j].vlines(wa, 0, p, color=cores[d], lw=3, label=nomes[d])
        print(f'   m-n={d:+d}: omega_a={wa:+.3f}  prob={p:.4f}')
    print('   soma =', round(soma, 5))
    ax[0, j].set_ylim(0, 1.05)
    ax[0, j].set_ylabel('probabilidade')
    ax[0, j].set_title(f'{titulo} ($\\omega={w:g}\\gamma$), $n={n}$: foton com cor exata')

    # ---- embaixo: pacote gaussiano de largura sig ----
    wa = np.linspace(xlim[0], xlim[1], 2001)
    total = np.zeros_like(wa)
    for d in dms:
        m = n + d
        nus = wa + d * w                     # cor de entrada que leva a saida wa
        Sm = xi2(nus) * np.array([abs(t_mn(m, n, v, w, g))**2 for v in nus])
        total += Sm
        ax[1, j].plot(wa, Sm, color=cores[d], lw=1.3)
    ax[1, j].plot(wa, xi2(wa), color='gray', ls=':', lw=1.2, label='espectro de entrada')
    ax[1, j].plot(wa, total, color='tab:green', ls='--', lw=1.5, label='espectro de saida total')
    ax[1, j].set_title(f'{titulo}: foton com largura $\\sigma={sig}\\gamma$')
    ax[1, j].set_xlabel(r'$\omega_a/\gamma$')
    ax[1, j].set_ylabel('densidade espectral')
    ax[1, j].set_xlim(*xlim)                 # mesmo eixo x em cima e embaixo

ax[0, 0].legend(fontsize=7.5, loc='upper left')
ax[1, 1].legend(fontsize=7.5, loc='upper right')
plt.tight_layout()
# (pacote.py salva aqui figura4_espectro_n{n}_sigma{sig}.svg; omitido para nao sobrescrever)
plt.show()

## 10. Como ler a figura

Os números abaixo são os dos parâmetros padrão ($n = 1$, $S = 0.25$, $\sigma = 0.3\gamma$, $\nu_0 = 0$).

### Mola rápida ($\omega = 3\gamma$)

**Em cima.** Os canais saem em cores bem separadas, $\omega_a = -d\,\omega = 3, 0, -3, -6, -9, -12$:

| $m-n$ | $\omega_a/\gamma$ | $P_m$ |
|---|---|---|
| $-1$ (anti-Stokes) | $+3$ | $0.047$ |
| $0$ (limpo) | $0$ | $0.582$ |
| $+1$ | $-3$ | $0.280$ |
| $+2$ | $-6$ | $0.078$ |
| $+3$ | $-9$ | $0.011$ |
| $+4$ | $-12$ | $0.001$ |

Só cerca de 58% dos fótons saem limpos. Os outros 42% saem com a cor trocada: a maioria cede energia à mola (linhas
vermelhas), e uma parte pequena recebe um quantum dela (linha azul). Essa linha azul só existe porque $n = 1$.

**Embaixo.** Cada linha vira um pico com aproximadamente a largura do pacote, centrado perto de $-d\,\omega$. O pico limpo
(preto) fica bem abaixo da gaussiana de entrada (cinza), porque a área que falta foi para as réplicas deslocadas. Os picos
também ficam **deformados**, já que $|t_{mn}(\nu)|^2$ varia dentro da largura $\sigma$. O pico anti-Stokes, por exemplo,
aparece em $\omega_a \approx 2.75$, e não em $3$. A curva total (verde tracejada) coincide com cada pico, porque os canais
não se sobrepõem.

### Mola lenta ($\omega = 0.1\gamma$)

**Em cima.** Os canais ficam a só $0.1\gamma$ uns dos outros. Com o acoplamento fraco ($g = 0.025\gamma$), o canal limpo
domina:

| $m-n$ | $\omega_a/\gamma$ | $P_m$ |
|---|---|---|
| $-1$ | $+0.1$ | $0.036$ |
| $0$ | $0$ | $0.892$ |
| $+1$ | $-0.1$ | $0.070$ |
| $+2$ | $-0.2$ | $0.002$ |

**Embaixo.** Como $\omega = 0.1\gamma < \sigma = 0.3\gamma$, os picos de todos os canais se sobrepõem. O espectro de saída total
fica praticamente igual ao de entrada, e a mudança de cor é invisível no espectro: só se veria medindo o estado da mola.

### Resumo

Com o **mesmo** $S$, a mola rápida separa espectralmente os fótons que trocaram energia com o banho. Na mola lenta, essa
"marca" do banho fica escondida dentro da largura do pacote.

## 11. Resumo das expressões

$$
\begin{aligned}
\beta &= \frac{2g}{\omega}, \qquad S = \beta^2, \qquad g = \frac{\sqrt{S}\,\omega}{2} \\[4pt]
F_{kn} &= \langle k|D(\beta)|n\rangle =
\begin{cases}
\sqrt{\dfrac{n!}{k!}}\,\beta^{k-n} e^{-\beta^2/2} L_n^{(k-n)}(\beta^2), & k \ge n \\[8pt]
(-1)^{n-k} F_{nk}, & k < n
\end{cases} \\[4pt]
t_{mn}(\nu) &= \delta_{mn} - \gamma \sum_{k=0}^{K-1}
\frac{F_{km} F_{kn}}{\gamma/2 + i\left(\Delta - 4g^2/\omega + (k-n)\omega - \nu\right)} \\[4pt]
\omega_a &= \nu - (m-n)\,\omega \qquad \text{(conservação de energia)} \\[4pt]
|\xi(\nu)|^2 &= \frac{1}{\sqrt{2\pi\sigma^2}}\,e^{-(\nu-\nu_0)^2/(2\sigma^2)}, \qquad \nu_0 = \Delta \\[4pt]
P_m &= |t_{mn}(\nu_0)|^2 \qquad \text{(linhas, fóton monocromático)} \\[4pt]
S_{\text{out}}(\omega_a) &= \sum_m \big|\xi(\omega_a + (m-n)\omega)\big|^2\,\big|t_{mn}(\omega_a + (m-n)\omega)\big|^2
\end{aligned}
$$